# 7.4 边缘AI监控与运维

边缘AI设备规模部署后，需要完善的监控和运维体系保障服务质量。与云端服务不同，边缘设备分布广、网络不稳定、资源受限，监控运维面临独特挑战。

## 为什么边缘AI需要监控？

1. **规模大**：百万级设备同时运行，手动运维不可行
2. **网络不稳定**：边缘设备可能长时间离线，需本地自治
3. **资源受限**：监控本身不能占用过多计算和存储资源
4. **模型漂移**：数据分布变化导致模型精度下降
5. **故障传播**：单设备故障可能影响整个服务链路

## 边缘AI监控的核心指标

| 类别 | 指标 | 采集频率 | 告警阈值 |
|------|------|---------|----------|
| **推理性能** | P50/P95/P99延迟 | 每次推理 | P99>500ms |
| **推理性能** | 吞吐量(req/s) | 每分钟 | <5 req/s |
| **模型质量** | 输出置信度 | 每次推理 | 均值<0.5 |
| **模型质量** | 用户反馈率 | 每小时 | 负反馈>10% |
| **资源使用** | CPU/GPU/NPU利用率 | 每10秒 | >90%持续5min |
| **资源使用** | 内存使用率 | 每10秒 | >85% |
| **系统健康** | 崩溃率 | 实时 | >0.1% |
| **网络** | 端云连接状态 | 每30秒 | 断连>5min |

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import time
import json
import hashlib
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Tuple
from collections import deque, OrderedDict
from enum import Enum

torch.manual_seed(42)
np.random.seed(42)
print(f"PyTorch version: {torch.__version__}")

### 监控指标体系

#### 什么是边缘AI监控指标体系？

边缘AI监控指标体系是分层采集、聚合、上报的指标集合，覆盖推理性能、模型质量、资源使用、系统健康四个维度。

#### 指标采集原则

1. **低开销**：监控代码不能影响推理延迟（<1%开销）
2. **本地聚合**：端侧聚合统计量（均值/分位数），减少上报数据量
3. **分级采集**：关键指标实时采集，次要指标周期采集

#### 指标聚合的数学原理

**滑动窗口分位数**：维护固定大小的滑动窗口，计算P50/P95/P99延迟：
$$P_k = \text{Percentile}(\{t_i\}_{i=n-W+1}^{n}, k)$$
其中$W$为窗口大小，$k$为分位数（50/95/99）。

**指数移动平均(EMA)**：平滑指标波动：
$$\bar{x}_t = \alpha \cdot x_t + (1 - \alpha) \cdot \bar{x}_{t-1}$$
典型$\alpha=0.1$，平滑近期波动。

In [ ]:
@dataclass
class MetricSample:
    name: str
    value: float
    timestamp: float
    tags: Dict[str, str] = field(default_factory=dict)

class SlidingWindowMetrics:
    """滑动窗口指标采集器: 支持分位数和EMA计算"""
    def __init__(self, window_size=1000, ema_alpha=0.1):
        self.window_size = window_size
        self.ema_alpha = ema_alpha
        self.samples: Dict[str, deque] = {}
        self.ema_values: Dict[str, float] = {}
        self.counters: Dict[str, float] = {}

    def record(self, name: str, value: float):
        """记录一个指标值"""
        if name not in self.samples:
            self.samples[name] = deque(maxlen=self.window_size)
        self.samples[name].append(value)
        if name in self.ema_values:
            self.ema_values[name] = self.ema_alpha * value + (1 - self.ema_alpha) * self.ema_values[name]
        else:
            self.ema_values[name] = value
        self.counters[name] = self.counters.get(name, 0) + 1

    def increment(self, name: str, value: float = 1.0):
        """递增计数器"""
        self.counters[name] = self.counters.get(name, 0) + value

    def get_percentile(self, name: str, percentile: float) -> Optional[float]:
        """计算分位数"""
        if name not in self.samples or len(self.samples[name]) == 0:
            return None
        values = np.array(self.samples[name])
        return float(np.percentile(values, percentile))

    def get_ema(self, name: str) -> Optional[float]:
        return self.ema_values.get(name)

    def get_summary(self, name: str) -> Dict:
        """获取指标摘要"""
        if name not in self.samples or len(self.samples[name]) == 0:
            return {}
        values = np.array(self.samples[name])
        return {
            'count': len(values), 'mean': float(np.mean(values)),
            'std': float(np.std(values)), 'min': float(np.min(values)),
            'max': float(np.max(values)), 'p50': float(np.percentile(values, 50)),
            'p95': float(np.percentile(values, 95)), 'p99': float(np.percentile(values, 99)),
            'ema': self.ema_values.get(name, 0),
        }

class EdgeMonitor:
    """边缘AI监控器: 采集推理性能、模型质量、资源使用指标"""
    def __init__(self, device_id: str = 'edge_001'):
        self.device_id = device_id
        self.metrics = SlidingWindowMetrics(window_size=1000, ema_alpha=0.1)
        self.inference_count = 0
        self.error_count = 0
        self.start_time = time.time()

    def record_inference(self, latency_ms: float, confidence: float, tokens_generated: int = 0):
        """记录一次推理"""
        self.inference_count += 1
        self.metrics.record('inference_latency_ms', latency_ms)
        self.metrics.record('inference_confidence', confidence)
        if tokens_generated > 0:
            self.metrics.record('tokens_per_second', tokens_generated / (latency_ms / 1000))

    def record_error(self, error_type: str):
        """记录一次错误"""
        self.error_count += 1
        self.metrics.increment(f'error_{error_type}')

    def record_resource(self, cpu_pct: float, mem_pct: float, npu_pct: float = 0):
        """记录资源使用"""
        self.metrics.record('cpu_usage_pct', cpu_pct)
        self.metrics.record('mem_usage_pct', mem_pct)
        if npu_pct > 0:
            self.metrics.record('npu_usage_pct', npu_pct)

    def get_health_report(self) -> Dict:
        """获取设备健康报告"""
        uptime_s = time.time() - self.start_time
        latency_summary = self.metrics.get_summary('inference_latency_ms')
        conf_summary = self.metrics.get_summary('inference_confidence')
        error_rate = self.error_count / max(self.inference_count, 1)
        return {
            'device_id': self.device_id, 'uptime_s': uptime_s,
            'inference_count': self.inference_count, 'error_count': self.error_count,
            'error_rate': error_rate, 'latency': latency_summary,
            'confidence': conf_summary,
        }

monitor = EdgeMonitor('edge_001')

np.random.seed(42)
for _ in range(200):
    latency = np.random.exponential(80) + 20
    confidence = np.random.beta(8, 2)
    monitor.record_inference(latency, confidence, tokens_generated=np.random.randint(5, 30))

for _ in range(3):
    monitor.record_error('timeout')
monitor.record_error('oom')

for _ in range(50):
    monitor.record_resource(np.random.uniform(30, 80), np.random.uniform(50, 85), np.random.uniform(20, 70))

report = monitor.get_health_report()
print(f"=== 边缘AI监控报告: {report['device_id']} ===")
print(f"运行时间: {report['uptime_s']:.0f}s")
print(f"推理次数: {report['inference_count']}, 错误率: {report['error_rate']:.2%}")

print(f"\n--- 推理延迟(ms) ---")
lat = report['latency']
print(f"  均值: {lat['mean']:.1f}, P50: {lat['p50']:.1f}, P95: {lat['p95']:.1f}, P99: {lat['p99']:.1f}")
print(f"  EMA: {lat['ema']:.1f}, 标准差: {lat['std']:.1f}")

print(f"\n--- 模型置信度 ---")
conf = report['confidence']
print(f"  均值: {conf['mean']:.3f}, P50: {conf['p50']:.3f}, 最低: {conf['min']:.3f}")

print(f"\n--- 资源使用 ---")
for name in ['cpu_usage_pct', 'mem_usage_pct', 'npu_usage_pct']:
    s = monitor.metrics.get_summary(name)
    if s:
        print(f"  {name}: 均值={s['mean']:.1f}%, P95={s['p95']:.1f}%")

print(f"\n--- 错误统计 ---")
for key, val in monitor.metrics.counters.items():
    if key.startswith('error_'):
        print(f"  {key}: {int(val)}")

---
### 数据上报策略

#### 为什么需要数据上报策略？

边缘设备网络不稳定、带宽有限，不能像云端服务那样实时上报所有指标。需要智能的上报策略：重要数据优先上报，普通数据聚合后周期上报。

#### 上报策略分类

| 策略 | 触发条件 | 数据量 | 实时性 | 适用场景 |
|------|---------|--------|--------|----------|
| **实时上报** | 告警/错误事件 | 小 | 高(ms级) | 故障告警 |
| **聚合上报** | 固定时间窗口 | 中 | 中(分钟级) | 性能指标 |
| **批量上报** | 网络可用时 | 大 | 低(小时级) | 日志/统计 |
| **压缩上报** | 带宽受限时 | 小 | 中 | 弱网环境 |
| **本地优先** | 离线时本地存储 | 无上报 | 离线 | 断网场景 |

#### 上报策略的数学原理

**自适应上报间隔**：根据网络质量和指标重要性动态调整：
$$T_{\text{report}} = T_{\text{base}} \cdot \frac{\text{BW}_{\text{current}}}{\text{BW}_{\text{target}}} \cdot \frac{1}{\text{priority}}$$

- $T_{\text{base}}$：基础上报间隔（如60秒）
- $\text{BW}_{\text{current}} / \text{BW}_{\text{target}}$：带宽比，带宽低则间隔长
- $\text{priority}$：指标优先级（告警=10, 性能=5, 统计=1）

In [ ]:
class ReportPriority(Enum):
    CRITICAL = 10
    HIGH = 5
    NORMAL = 2
    LOW = 1

@dataclass
class ReportItem:
    metric_name: str
    value: float
    timestamp: float
    priority: ReportPriority
    aggregated: bool = False
    tags: Dict[str, str] = field(default_factory=dict)

class AdaptiveReportScheduler:
    """自适应上报调度器: 根据网络条件和优先级调度上报"""
    def __init__(self, base_interval_s=60, target_bw_kbps=100):
        self.base_interval_s = base_interval_s
        self.target_bw_kbps = target_bw_kbps
        self.current_bw_kbps = target_bw_kbps
        self.pending_items: List[ReportItem] = []
        self.local_storage: List[ReportItem] = []
        self.reported_count = 0
        self.dropped_count = 0
        self.last_report_time = 0

    def update_bandwidth(self, bw_kbps: float):
        self.current_bw_kbps = max(bw_kbps, 1)

    def compute_report_interval(self, priority: ReportPriority) -> float:
        """计算自适应上报间隔"""
        bw_ratio = self.target_bw_kbps / self.current_bw_kbps
        return self.base_interval_s * bw_ratio / priority.value

    def add_item(self, item: ReportItem):
        """添加待上报数据"""
        if item.priority == ReportPriority.CRITICAL:
            self.pending_items.insert(0, item)
        else:
            self.pending_items.append(item)

    def aggregate_window(self, window_s: float = 300) -> List[ReportItem]:
        """聚合时间窗口内的指标"""
        now = time.time()
        to_aggregate = [item for item in self.pending_items
                        if not item.priority == ReportPriority.CRITICAL
                        and now - item.timestamp < window_s]
        critical = [item for item in self.pending_items if item.priority == ReportPriority.CRITICAL]
        aggregated = []
        groups = {}
        for item in to_aggregate:
            key = item.metric_name
            if key not in groups:
                groups[key] = []
            groups[key].append(item)
        for name, items in groups.items():
            values = [i.value for i in items]
            aggregated.append(ReportItem(
                metric_name=f'{name}_agg',
                value=float(np.mean(values)),
                timestamp=now, priority=items[0].priority,
                aggregated=True,
                tags={'count': str(len(values)), 'min': str(min(values)), 'max': str(max(values))}
            ))
        return critical + aggregated

    def flush(self, network_available: bool = True) -> Dict:
        """执行上报"""
        if not network_available:
            self.local_storage.extend(self.pending_items)
            stored = len(self.pending_items)
            self.pending_items = []
            return {'action': 'stored_locally', 'items_stored': stored}
        items = self.aggregate_window()
        reported = len(items)
        self.reported_count += reported
        self.pending_items = []
        self.last_report_time = time.time()
        return {'action': 'reported', 'items_reported': reported}

scheduler = AdaptiveReportScheduler(base_interval_s=60, target_bw_kbps=100)

print("=== 自适应数据上报策略 ===")
print(f"\n--- 不同优先级和带宽的上报间隔 ---")
print(f"{'优先级':<12} {'100kbps':<12} {'50kbps':<12} {'10kbps':<12} {'1kbps'}")
print("-" * 60)
for priority in ReportPriority:
    intervals = []
    for bw in [100, 50, 10, 1]:
        scheduler.update_bandwidth(bw)
        interval = scheduler.compute_report_interval(priority)
        intervals.append(f"{interval:.0f}s")
    print(f"{priority.name:<12} {intervals[0]:<12} {intervals[1]:<12} {intervals[2]:<12} {intervals[3]}")

scheduler.update_bandwidth(100)
now = time.time()
for i in range(50):
    scheduler.add_item(ReportItem('latency_ms', np.random.exponential(80)+20, now-i, ReportPriority.NORMAL))
for i in range(5):
    scheduler.add_item(ReportItem('error_rate', 0.15, now-i, ReportPriority.HIGH))
scheduler.add_item(ReportItem('crash', 1.0, now, ReportPriority.CRITICAL))

print(f"\n--- 上报模拟 ---")
print(f"待上报数据: {len(scheduler.pending_items)}条")
result = scheduler.flush(network_available=True)
print(f"上报结果: {result}")

for i in range(20):
    scheduler.add_item(ReportItem('latency_ms', np.random.exponential(80)+20, now-i, ReportPriority.NORMAL))
result = scheduler.flush(network_available=False)
print(f"\n离线上报: {result}")
print(f"本地存储: {len(scheduler.local_storage)}条")

print(f"\n=== 产业实践要点 ===")
print(f"1. 告警实时上报: CRITICAL优先级立即上报, 不等待聚合")
print(f"2. 指标聚合上报: NORMAL优先级5分钟窗口聚合后上报")
print(f"3. 离线本地存储: 断网时数据存本地, 恢复后批量上报")
print(f"4. 带宽自适应: 弱网时增大上报间隔, 减少数据量")
print(f"5. 数据压缩: 上报前gzip压缩, 减少50-80%传输量")

---
### 告警与故障响应

#### 什么是告警与故障响应？

当监控指标超过阈值或发生异常时，自动触发告警并执行预定义的响应策略，减少人工干预。

#### 告警级别

| 级别 | 触发条件 | 响应策略 | 通知方式 |
|------|---------|---------|----------|
| **P0-紧急** | 服务完全不可用 | 自动降级+人工介入 | 电话+短信+IM |
| **P1-严重** | 核心指标严重超标 | 自动降级 | 短信+IM |
| **P2-警告** | 指标持续偏高 | 自动调整参数 | IM |
| **P3-提示** | 轻微异常 | 记录日志 | 日报 |

#### 自动降级策略

```
正常 → P2(性能下降) → 切换小模型/降低精度
     → P1(严重超标) → 端侧独立推理，断开云端
     → P0(服务不可用) → 返回缓存结果/默认回复
```

In [ ]:
class AlertSeverity(Enum):
    P0_CRITICAL = 0
    P1_MAJOR = 1
    P2_WARNING = 2
    P3_INFO = 3

@dataclass
class AlertRule:
    name: str
    metric_name: str
    condition: str
    threshold: float
    duration_s: float
    severity: AlertSeverity
    action: str

@dataclass
class Alert:
    rule_name: str
    severity: AlertSeverity
    message: str
    timestamp: float
    action_taken: str
    resolved: bool = False

class AlertManager:
    """告警管理器: 规则匹配+自动响应"""
    def __init__(self):
        self.rules: List[AlertRule] = []
        self.active_alerts: List[Alert] = []
        self.alert_history: List[Alert] = []
        self.metric_violations: Dict[str, float] = {}

    def add_rule(self, rule: AlertRule):
        self.rules.append(rule)

    def evaluate(self, metrics: Dict[str, float]) -> List[Alert]:
        """评估所有规则，生成告警"""
        new_alerts = []
        now = time.time()
        for rule in self.rules:
            value = metrics.get(rule.metric_name)
            if value is None:
                continue
            violated = False
            if rule.condition == 'gt' and value > rule.threshold:
                violated = True
            elif rule.condition == 'lt' and value < rule.threshold:
                violated = True
            key = f"{rule.metric_name}_{rule.condition}_{rule.threshold}"
            if violated:
                if key not in self.metric_violations:
                    self.metric_violations[key] = now
                duration = now - self.metric_violations[key]
                if duration >= rule.duration_s:
                    alert = Alert(
                        rule_name=rule.name, severity=rule.severity,
                        message=f"{rule.metric_name}={value:.2f} {rule.condition} {rule.threshold} (持续{duration:.0f}s)",
                        timestamp=now, action_taken=rule.action
                    )
                    new_alerts.append(alert)
                    self.active_alerts.append(alert)
                    self.metric_violations.pop(key, None)
            else:
                self.metric_violations.pop(key, None)
        return new_alerts

    def resolve_alert(self, alert: Alert):
        alert.resolved = True
        self.active_alerts.remove(alert)
        self.alert_history.append(alert)

    def get_active_alerts_summary(self) -> Dict:
        severity_counts = {s: 0 for s in AlertSeverity}
        for alert in self.active_alerts:
            severity_counts[alert.severity] += 1
        return {'total_active': len(self.active_alerts), 'by_severity': severity_counts}

alert_mgr = AlertManager()
alert_mgr.add_rule(AlertRule('high_latency', 'p99_latency_ms', 'gt', 500, 60, AlertSeverity.P1_MAJOR, 'switch_to_small_model'))
alert_mgr.add_rule(AlertRule('low_confidence', 'avg_confidence', 'lt', 0.5, 120, AlertSeverity.P2_WARNING, 'flag_for_review'))
alert_mgr.add_rule(AlertRule('high_error_rate', 'error_rate', 'gt', 0.05, 30, AlertSeverity.P0_CRITICAL, 'fallback_to_cache'))
alert_mgr.add_rule(AlertRule('high_memory', 'mem_usage_pct', 'gt', 85, 300, AlertSeverity.P2_WARNING, 'reduce_batch_size'))
alert_mgr.add_rule(AlertRule('low_throughput', 'throughput_rps', 'lt', 5, 120, AlertSeverity.P3_INFO, 'log_and_monitor'))

print("=== 告警规则配置 ===")
print(f"\n{'规则名':<18} {'指标':<18} {'条件':<8} {'阈值':<10} {'持续':<8} {'级别':<6} {'动作'}")
print("-" * 85)
for rule in alert_mgr.rules:
    print(f"{rule.name:<18} {rule.metric_name:<18} {rule.condition:<8} {rule.threshold:<10} "
          f"{rule.duration_s:<8.0f}s {rule.severity.name:<6} {rule.action}")

print(f"\n--- 告警触发模拟 ---")
test_metrics_list = [
    {'p99_latency_ms': 450, 'avg_confidence': 0.7, 'error_rate': 0.02, 'mem_usage_pct': 70, 'throughput_rps': 10},
    {'p99_latency_ms': 600, 'avg_confidence': 0.4, 'error_rate': 0.02, 'mem_usage_pct': 70, 'throughput_rps': 10},
    {'p99_latency_ms': 550, 'avg_confidence': 0.3, 'error_rate': 0.08, 'mem_usage_pct': 90, 'throughput_rps': 3},
]
for i, metrics in enumerate(test_metrics_list):
    time.sleep(0.01)
    for key in alert_mgr.metric_violations:
        alert_mgr.metric_violations[key] = time.time() - 120
    alerts = alert_mgr.evaluate(metrics)
    if alerts:
        print(f"\n  场景{i+1}: {metrics}")
        for alert in alerts:
            print(f"    [{alert.severity.name}] {alert.message} → 动作: {alert.action_taken}")

print(f"\n=== 产业实践要点 ===")
print(f"1. 告警必须有持续时间阈值, 避免瞬时波动误报")
print(f"2. P0告警必须自动降级, 不能等待人工介入")
print(f"3. 告警收敛: 同类告警合并, 避免告警风暴")
print(f"4. 降级策略: 大模型→小模型→缓存回复, 渐进降级")
print(f"5. 告警升级: P2持续30min→升级为P1, P1持续15min→升级为P0")

---
### A/B测试与模型版本管理

#### 什么是端侧A/B测试？

端侧A/B测试是将不同版本的模型部署到不同用户群体，对比关键指标（延迟、精度、用户满意度），决定是否全量发布。

#### A/B测试的关键指标

| 指标 | 对照组(A) | 实验组(B) | 判定标准 |
|------|----------|----------|----------|
| P99延迟 | 450ms | 380ms | B比A低15%+ |
| 模型精度 | 92.1% | 91.8% | B不低于A 0.5% |
| 用户满意度 | 4.2/5 | 4.3/5 | B不低于A |
| 崩溃率 | 0.05% | 0.04% | B不高于A |

#### 统计显著性

A/B测试需要足够的样本量才能得出统计显著的结论：
$$n = \frac{(Z_{\alpha/2} + Z_\beta)^2 \cdot 2\sigma^2}{\delta^2}$$
其中$\delta$为最小可检测效应(MDE)，$\sigma$为指标标准差，$Z_{\alpha/2}=1.96$（95%置信度），$Z_\beta=0.84$（80%统计功效）。

#### 模型版本管理

| 版本 | 描述 | 状态 | 流量比例 |
|------|------|------|----------|
| v1.0 | 基线模型 | 稳定 | 90% |
| v1.1-rc | 优化延迟 | 灰度 | 10% |
| v1.2-beta | 新功能 | 测试 | 内测 |

In [ ]:
@dataclass
class ModelVersion:
    version: str
    model_hash: str
    description: str
    status: str
    traffic_pct: float
    created_at: float

class ABTestFramework:
    """A/B测试框架: 模型版本管理+流量分配+统计检验"""
    def __init__(self):
        self.versions: Dict[str, ModelVersion] = {}
        self.experiments: Dict[str, Dict] = {}
        self.results: Dict[str, Dict] = {}

    def register_version(self, version: str, model_hash: str, description: str, status: str, traffic_pct: float):
        self.versions[version] = ModelVersion(version, model_hash, description, status, traffic_pct, time.time())

    def create_experiment(self, name: str, control_version: str, treatment_version: str, traffic_split: float = 0.1):
        """创建A/B测试实验"""
        self.experiments[name] = {
            'control': control_version, 'treatment': treatment_version,
            'traffic_split': traffic_split, 'start_time': time.time(),
            'control_metrics': [], 'treatment_metrics': [],
        }

    def record_metric(self, experiment_name: str, version: str, metric_name: str, value: float):
        """记录实验指标"""
        if experiment_name not in self.experiments:
            return
        exp = self.experiments[experiment_name]
        group = 'control' if version == exp['control'] else 'treatment'
        exp[f'{group}_metrics'].append({metric_name: value, 'timestamp': time.time()})

    def analyze_experiment(self, experiment_name: str, metric_name: str) -> Dict:
        """分析实验结果: 均值对比+统计显著性"""
        if experiment_name not in self.experiments:
            return {}
        exp = self.experiments[experiment_name]
        control_values = [m[metric_name] for m in exp['control_metrics'] if metric_name in m]
        treatment_values = [m[metric_name] for m in exp['treatment_metrics'] if metric_name in m]
        if len(control_values) < 10 or len(treatment_values) < 10:
            return {'status': 'insufficient_data', 'control_n': len(control_values), 'treatment_n': len(treatment_values)}
        control_mean = np.mean(control_values)
        treatment_mean = np.mean(treatment_values)
        control_std = np.std(control_values)
        treatment_std = np.std(treatment_values)
        diff_pct = (treatment_mean - control_mean) / abs(control_mean) * 100
        pooled_std = np.sqrt((control_std**2 + treatment_std**2) / 2)
        if pooled_std > 0:
            z_score = (treatment_mean - control_mean) / (pooled_std * np.sqrt(2 / len(control_values)))
            p_value = 2 * (1 - 0.5 * (1 + np.math.erf(abs(z_score) / np.sqrt(2))))
        else:
            z_score = 0
            p_value = 1.0
        return {
            'status': 'complete', 'control_mean': control_mean, 'treatment_mean': treatment_mean,
            'control_std': control_std, 'treatment_std': treatment_std,
            'diff_pct': diff_pct, 'z_score': z_score, 'p_value': p_value,
            'significant': p_value < 0.05, 'control_n': len(control_values), 'treatment_n': len(treatment_values),
        }

ab = ABTestFramework()
ab.register_version('v1.0', 'hash_a1b2', '基线模型(INT4)', 'stable', 90)
ab.register_version('v1.1-rc', 'hash_c3d4', '优化延迟版本(INT4+算子融合)', 'canary', 10)
ab.create_experiment('latency_optimization', 'v1.0', 'v1.1-rc', traffic_split=0.1)

np.random.seed(42)
for _ in range(200):
    ab.record_metric('latency_optimization', 'v1.0', 'p99_latency_ms', np.random.exponential(80) + 350)
    ab.record_metric('latency_optimization', 'v1.1-rc', 'p99_latency_ms', np.random.exponential(60) + 280)
for _ in range(200):
    ab.record_metric('latency_optimization', 'v1.0', 'accuracy', np.random.normal(0.92, 0.02))
    ab.record_metric('latency_optimization', 'v1.1-rc', 'accuracy', np.random.normal(0.915, 0.02))

print("=== A/B测试分析 ===")
print(f"\n--- 模型版本 ---")
for v in ab.versions.values():
    print(f"  {v.version}: {v.description} ({v.status}, {v.traffic_pct}%流量)")

for metric in ['p99_latency_ms', 'accuracy']:
    result = ab.analyze_experiment('latency_optimization', metric)
    print(f"\n--- {metric} 对比 ---")
    if result.get('status') == 'complete':
        print(f"  对照组(v1.0): 均值={result['control_mean']:.2f}, 标准差={result['control_std']:.2f}, n={result['control_n']}")
        print(f"  实验组(v1.1): 均值={result['treatment_mean']:.2f}, 标准差={result['treatment_std']:.2f}, n={result['treatment_n']}")
        print(f"  差异: {result['diff_pct']:+.1f}%")
        print(f"  Z-score: {result['z_score']:.2f}, P-value: {result['p_value']:.4f}")
        print(f"  统计显著: {'✓ 是' if result['significant'] else '✗ 否'} (p<0.05)")
    else:
        print(f"  数据不足: {result}")

print(f"\n=== 产业实践要点 ===")
print(f"1. 灰度发布: 新模型先1%流量, 逐步增加到10%→50%→100%")
print(f"2. 样本量: 至少1000次推理才能得出统计显著的结论")
print(f"3. 护栏指标: 延迟可以优化, 但精度不能下降>0.5%")
print(f"4. 自动回滚: 实验组崩溃率>0.1%时自动回滚到对照组")
print(f"5. 实验隔离: 同一用户始终在同一组, 避免体验不一致")

---
### OTA更新与模型版本管理

#### 什么是OTA更新？

Over-The-Air更新是远程向边缘设备推送模型或软件更新的机制。端侧AI的OTA需要特别关注更新安全、回滚策略和带宽优化。

#### OTA更新的关键挑战

| 挑战 | 原因 | 解决方案 |
|------|------|----------|
| **更新安全** | 恶意更新可能植入后门 | 签名校验+TEE验证 |
| **回滚风险** | 新模型可能有问题 | A/B分区+自动回滚 |
| **带宽限制** | 模型文件大(GB级) | 增量更新+压缩 |
| **断网恢复** | 更新中断 | 断点续传+校验 |
| **版本碎片** | 设备运行不同版本 | 最低版本要求+强制更新 |

#### 增量更新原理

仅传输模型参数的差异部分：
$$\Delta W = W_{\text{new}} - W_{\text{old}}$$
端侧重建：$W_{\text{new}} = W_{\text{old}} + \Delta W$

压缩比：$\frac{|\Delta W|_{\text{compressed}}}{|W_{\text{new}}|}$，典型5-20%（微调模型参数变化小）

In [ ]:
@dataclass
class OTAUpdate:
    update_id: str
    target_version: str
    model_hash: str
    size_bytes: int
    delta_size_bytes: int
    signature: str
    created_at: float
    is_delta: bool = True

@dataclass
class OTAResult:
    update_id: str
    success: bool
    integrity_ok: bool
    rollback_needed: bool
    duration_s: float
    message: str

class OTAManager:
    """OTA更新管理器: 增量更新+签名校验+自动回滚"""
    def __init__(self, current_version: str = 'v1.0'):
        self.current_version = current_version
        self.current_model_hash = 'hash_a1b2'
        self.backup_version = None
        self.backup_hash = None
        self.update_history: List[OTAResult] = []

    def _verify_signature(self, update: OTAUpdate) -> bool:
        expected = hashlib.sha256(f"{update.target_version}_{update.model_hash}".encode()).hexdigest()[:16]
        return update.signature == expected

    def _compute_delta(self, old_params: Dict[str, torch.Tensor], new_params: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        """计算参数增量"""
        delta = OrderedDict()
        for key in new_params:
            if key in old_params:
                delta[key] = new_params[key] - old_params[key]
            else:
                delta[key] = new_params[key]
        return delta

    def _apply_delta(self, old_params: Dict[str, torch.Tensor], delta: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        """应用增量更新"""
        new_params = OrderedDict()
        for key in old_params:
            if key in delta:
                new_params[key] = old_params[key] + delta[key]
            else:
                new_params[key] = old_params[key]
        for key in delta:
            if key not in old_params:
                new_params[key] = delta[key]
        return new_params

    def create_update(self, old_model: nn.Module, new_model: nn.Module, target_version: str) -> OTAUpdate:
        """创建增量更新包"""
        old_params = old_model.state_dict()
        new_params = new_model.state_dict()
        delta = self._compute_delta(old_params, new_params)
        full_size = sum(p.numel() * p.element_size() for p in new_params.values())
        delta_size = sum(p.numel() * p.element_size() for p in delta.values())
        model_hash = hashlib.sha256(str(sorted(new_params.keys())).encode()).hexdigest()[:16]
        signature = hashlib.sha256(f"{target_version}_{model_hash}".encode()).hexdigest()[:16]
        return OTAUpdate(
            update_id=f"ota_{target_version}_{int(time.time())}",
            target_version=target_version, model_hash=model_hash,
            size_bytes=full_size, delta_size_bytes=delta_size,
            signature=signature, created_at=time.time(), is_delta=True
        )

    def apply_update(self, update: OTAUpdate, old_model: nn.Module, new_model: nn.Module,
                     auto_rollback: bool = True) -> OTAResult:
        """应用OTA更新"""
        start = time.time()
        if not self._verify_signature(update):
            return OTAResult(update.update_id, False, False, False, time.time()-start, '签名校验失败')
        self.backup_version = self.current_version
        self.backup_hash = self.current_model_hash
        old_params = old_model.state_dict()
        new_params = new_model.state_dict()
        delta = self._compute_delta(old_params, new_params)
        reconstructed = self._apply_delta(old_params, delta)
        for key in new_params:
            if key in reconstructed:
                max_diff = (new_params[key] - reconstructed[key]).abs().max().item()
                if max_diff > 1e-6:
                    if auto_rollback:
                        return OTAResult(update.update_id, False, True, True, time.time()-start,
                                         f'增量重建失败: {key}差异={max_diff}')
        self.current_version = update.target_version
        self.current_model_hash = update.model_hash
        return OTAResult(update.update_id, True, True, False, time.time()-start, '更新成功')

    def rollback(self) -> str:
        """回滚到上一版本"""
        if self.backup_version:
            self.current_version = self.backup_version
            self.current_model_hash = self.backup_hash
            return f'已回滚到 {self.backup_version}'
        return '无可用备份版本'

ota = OTAManager('v1.0')
old_model = nn.Sequential(nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 5))
new_model = nn.Sequential(nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 5))
with torch.no_grad():
    for p in new_model.parameters():
        p += torch.randn_like(p) * 0.01

update = ota.create_update(old_model, new_model, 'v1.1')

print("=== OTA更新管理 ===")
print(f"\n--- 更新包信息 ---")
print(f"  目标版本: {update.target_version}")
print(f"  全量大小: {update.size_bytes/1024:.1f} KB")
print(f"  增量大小: {update.delta_size_bytes/1024:.1f} KB")
print(f"  压缩比: {update.delta_size_bytes/update.size_bytes:.1%}")
print(f"  签名: {update.signature}")

result = ota.apply_update(update, old_model, new_model)
print(f"\n--- 更新结果 ---")
print(f"  成功: {'✓' if result.success else '✗'}")
print(f"  完整性: {'✓' if result.integrity_ok else '✗'}")
print(f"  消息: {result.message}")
print(f"  当前版本: {ota.current_version}")

print(f"\n--- 回滚测试 ---")
rollback_msg = ota.rollback()
print(f"  {rollback_msg}")
print(f"  当前版本: {ota.current_version}")

print(f"\n--- 增量更新压缩效果 ---")
for change_scale in [0.001, 0.01, 0.05, 0.1]:
    test_new = nn.Sequential(nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 5))
    with torch.no_grad():
        for p in test_new.parameters():
            p += torch.randn_like(p) * change_scale
    u = ota.create_update(old_model, test_new, f'test_{change_scale}')
    ratio = u.delta_size_bytes / u.size_bytes
    sparse_pct = sum(1 for p in (test_new.state_dict().values()) if True) / len(list(test_new.state_dict().values()))
    print(f"  参数变化σ={change_scale}: 增量压缩比={ratio:.1%}")

print(f"\n=== 产业实践要点 ===")
print(f"1. A/B分区: 两个模型分区, 更新写入备用分区, 失败回滚")
print(f"2. 签名校验: RSA-2048签名, 防止恶意更新")
print(f"3. 增量更新: 微调模型参数变化小, 增量包仅为全量的5-20%")
print(f"4. 灰度推送: 先1%设备, 验证无问题后逐步全量")
print(f"5. 自动回滚: 更新后崩溃率>0.5%自动回滚到旧版本")
print(f"6. 断点续传: 大模型下载中断后可从断点继续")

---
### 日志分析与性能回归检测

#### 为什么需要日志分析？

边缘设备产生的日志是排查问题、检测性能回归的主要数据源。但日志量大、格式不统一，需要结构化采集和智能分析。

#### 日志分类

| 类型 | 内容 | 采集频率 | 存储周期 |
|------|------|---------|----------|
| **推理日志** | 输入摘要+输出+延迟+置信度 | 每次推理 | 7天 |
| **系统日志** | CPU/内存/网络状态 | 每10秒 | 3天 |
| **错误日志** | 异常堆栈+上下文 | 实时 | 30天 |
| **审计日志** | 模型版本+配置变更 | 事件触发 | 1年 |

#### 性能回归检测

性能回归是指模型更新或环境变化后，关键指标（延迟、精度）显著恶化。检测方法：

1. **基线对比**：与上一版本的基线指标对比
2. **趋势检测**：检测指标的时间趋势是否异常
3. **异常检测**：使用统计方法检测异常点

**CUSUM算法**（累积和）：检测均值漂移
$$S_t = \max(0, S_{t-1} + (x_t - \mu_0 - k))$$
当$S_t > h$时触发告警。$k$为允许偏移量，$h$为告警阈值。

In [ ]:
@dataclass
class LogEntry:
    timestamp: float
    level: str
    category: str
    message: str
    metadata: Dict[str, float] = field(default_factory=dict)

class StructuredLogger:
    """结构化日志采集器"""
    def __init__(self, device_id: str, max_entries: int = 10000):
        self.device_id = device_id
        self.max_entries = max_entries
        self.entries: List[LogEntry] = []
        self.counters: Dict[str, int] = {}

    def log(self, level: str, category: str, message: str, metadata: Dict[str, float] = None):
        entry = LogEntry(time.time(), level, category, message, metadata or {})
        self.entries.append(entry)
        if len(self.entries) > self.max_entries:
            self.entries = self.entries[-self.max_entries:]
        key = f"{level}_{category}"
        self.counters[key] = self.counters.get(key, 0) + 1

    def query(self, level: str = None, category: str = None, last_n: int = 100) -> List[LogEntry]:
        results = self.entries
        if level:
            results = [e for e in results if e.level == level]
        if category:
            results = [e for e in results if e.category == category]
        return results[-last_n:]

class CUSUMDetector:
    """CUSUM异常检测器: 检测性能回归"""
    def __init__(self, baseline_mean: float, k: float = 0.5, h: float = 5.0):
        self.baseline_mean = baseline_mean
        self.k = k
        self.h = h
        self.s_pos = 0.0
        self.s_neg = 0.0
        self.history = []

    def update(self, value: float) -> Dict:
        """更新CUSUM统计量"""
        self.s_pos = max(0, self.s_pos + (value - self.baseline_mean - self.k))
        self.s_neg = max(0, self.s_neg + (self.baseline_mean - value - self.k))
        alert = self.s_pos > self.h or self.s_neg > self.h
        direction = 'increase' if self.s_pos > self.h else 'decrease' if self.s_neg > self.h else None
        result = {
            'value': value, 's_pos': self.s_pos, 's_neg': self.s_neg,
            'alert': alert, 'direction': direction,
        }
        self.history.append(result)
        return result

    def reset(self):
        self.s_pos = 0.0
        self.s_neg = 0.0
        self.history = []

logger = StructuredLogger('edge_001')
np.random.seed(42)

for i in range(50):
    latency = np.random.normal(80, 10)
    level = 'INFO' if latency < 100 else 'WARN' if latency < 150 else 'ERROR'
    logger.log(level, 'inference', f'latency={latency:.1f}ms', {'latency_ms': latency, 'confidence': np.random.beta(8, 2)})

logger.log('ERROR', 'system', 'OOM detected', {'mem_usage_pct': 92.5})
logger.log('WARN', 'network', 'Cloud connection timeout', {'rtt_ms': 500})
logger.log('INFO', 'model', 'Model v1.1 loaded', {'model_size_mb': 3800})

print("=== 结构化日志分析 ===")
print(f"\n--- 日志统计 ---")
for key, count in sorted(logger.counters.items()):
    print(f"  {key}: {count}")

errors = logger.query(level='ERROR')
print(f"\n--- 错误日志 ---")
for entry in errors:
    print(f"  [{entry.level}] {entry.category}: {entry.message}")
    if entry.metadata:
        print(f"    元数据: {entry.metadata}")

print(f"\n=== CUSUM性能回归检测 ===")
detector = CUSUMDetector(baseline_mean=80, k=1.0, h=10.0)
np.random.seed(42)

latencies_normal = np.random.normal(80, 8, 100)
latencies_regression = np.random.normal(120, 15, 50)
all_latencies = np.concatenate([latencies_normal, latencies_regression])

print(f"\n--- 检测过程 (基线均值=80ms, 回归在第100步引入) ---")
print(f"{'步数':<8} {'延迟(ms)':<12} {'S+':<10} {'S-':<10} {'告警':<8} {'方向'}")
print("-" * 55)
alerts = []
for i, val in enumerate(all_latencies):
    result = detector.update(val)
    if result['alert'] or i in [0, 50, 99, 100, 110, 120, 149]:
        alert_str = '⚠ 告警' if result['alert'] else ''
        direction = result['direction'] or ''
        print(f"{i:<8} {val:<12.1f} {result['s_pos']:<10.1f} {result['s_neg']:<10.1f} {alert_str:<8} {direction}")
        if result['alert']:
            alerts.append(i)

if alerts:
    print(f"\n检测到性能回归: 第{alerts[0]}步, 延迟从80ms基线上升到~120ms")
    print(f"CUSUM在回归发生后约{alerts[0]-100}步内检测到异常")

print(f"\n=== 产业实践要点 ===")
print(f"1. 结构化日志: 统一格式, 便于自动分析和查询")
print(f"2. CUSUM检测: 比简单阈值更灵敏, 能检测渐进式回归")
print(f"3. 日志脱敏: 推理日志不含原始输入, 仅记录摘要和指标")
print(f"4. 日志分级: ERROR实时上报, INFO聚合后周期上报")
print(f"5. 性能基线: 每个模型版本建立基线, 新版本与基线对比")
print(f"6. 自动回滚: 检测到性能回归后自动回滚到上一版本")

---
### 边缘设备集群监控

#### 什么是边缘设备集群监控？

管理大规模边缘设备集群的监控数据，聚合全局视角的指标，发现集群级别的问题。

#### 集群监控的关键指标

| 指标 | 计算方式 | 告警条件 |
|------|---------|----------|
| **在线率** | 在线设备数/总设备数 | <95% |
| **模型版本分布** | 各版本设备数占比 | 旧版本>20% |
| **全局P99延迟** | 所有设备P99的P99 | >1000ms |
| **全局错误率** | 总错误数/总推理数 | >1% |
| **异常设备比例** | 异常设备数/总设备数 | >5% |

In [ ]:
class FleetMonitor:
    """边缘设备集群监控器"""
    def __init__(self, n_devices: int = 100):
        self.n_devices = n_devices
        self.devices: Dict[str, Dict] = {}
        np.random.seed(42)
        for i in range(n_devices):
            device_id = f"edge_{i:04d}"
            self.devices[device_id] = {
                'online': np.random.random() > 0.03,
                'version': np.random.choice(['v1.0', 'v1.1', 'v1.2'], p=[0.3, 0.5, 0.2]),
                'p99_latency_ms': np.random.exponential(80) + 300,
                'error_rate': np.random.exponential(0.005),
                'cpu_usage_pct': np.random.uniform(30, 85),
                'mem_usage_pct': np.random.uniform(40, 90),
                'inference_count': np.random.randint(100, 10000),
            }

    def get_fleet_summary(self) -> Dict:
        """获取集群摘要"""
        online_count = sum(1 for d in self.devices.values() if d['online'])
        version_dist = {}
        for d in self.devices.values():
            v = d['version']
            version_dist[v] = version_dist.get(v, 0) + 1
        online_devices = [d for d in self.devices.values() if d['online']]
        p99_latencies = [d['p99_latency_ms'] for d in online_devices]
        error_rates = [d['error_rate'] for d in online_devices]
        abnormal = sum(1 for d in online_devices
                       if d['error_rate'] > 0.05 or d['p99_latency_ms'] > 800)
        return {
            'total_devices': self.n_devices, 'online_devices': online_count,
            'online_rate': online_count / self.n_devices,
            'version_distribution': version_dist,
            'global_p99_latency': float(np.percentile(p99_latencies, 99)) if p99_latencies else 0,
            'global_p50_latency': float(np.percentile(p99_latencies, 50)) if p99_latencies else 0,
            'global_error_rate': float(np.mean(error_rates)) if error_rates else 0,
            'abnormal_devices': abnormal,
            'abnormal_rate': abnormal / max(online_count, 1),
        }

    def get_device_details(self, device_id: str) -> Optional[Dict]:
        return self.devices.get(device_id)

fleet = FleetMonitor(n_devices=1000)
summary = fleet.get_fleet_summary()

print("=== 边缘设备集群监控 ===")
print(f"\n--- 集群概览 ---")
print(f"总设备数: {summary['total_devices']}")
print(f"在线设备: {summary['online_devices']} ({summary['online_rate']:.1%})")
print(f"异常设备: {summary['abnormal_devices']} ({summary['abnormal_rate']:.1%})")

print(f"\n--- 模型版本分布 ---")
for version, count in sorted(summary['version_distribution'].items()):
    pct = count / summary['total_devices'] * 100
    bar = '█' * int(pct / 2)
    print(f"  {version}: {count} ({pct:.0f}%) {bar}")

print(f"\n--- 全局性能 ---")
print(f"全局P50延迟: {summary['global_p50_latency']:.0f}ms")
print(f"全局P99延迟: {summary['global_p99_latency']:.0f}ms")
print(f"全局错误率: {summary['global_error_rate']:.2%}")

print(f"\n--- 单设备详情 ---")
for device_id in ['edge_0000', 'edge_0001', 'edge_0042']:
    d = fleet.get_device_details(device_id)
    if d:
        status = '在线' if d['online'] else '离线'
        print(f"  {device_id}: {status}, {d['version']}, P99={d['p99_latency_ms']:.0f}ms, "
              f"错误率={d['error_rate']:.2%}")

print(f"\n=== 产业实践要点 ===")
print(f"1. 集群视角: 单设备异常不重要, 集群级趋势才需要关注")
print(f"2. 版本收敛: 新版本发布后, 监控旧版本占比, 推动OTA更新")
print(f"3. 异常设备: 自动标记异常设备, 优先排查和修复")
print(f"4. 容量规划: 根据全局负载趋势, 提前扩容或缩容")
print(f"5. 地域分析: 按地域聚合指标, 发现区域性网络问题")
print(f"6. 仪表盘: Grafana/Prometheus构建实时监控仪表盘")

---
## 总结与最佳实践

### 边缘AI监控的核心原则

1. **本地自治优先**：端侧能自行处理的告警和降级，不依赖云端
2. **数据分级上报**：告警实时上报，指标聚合上报，日志批量上报
3. **渐进降级**：大模型→小模型→缓存回复，确保服务可用
4. **灰度发布**：新模型先1%流量验证，通过后逐步全量

### 监控运维Checklist

- [ ] 部署监控指标采集（延迟/置信度/资源使用）
- [ ] 配置自适应数据上报策略（实时/聚合/批量）
- [ ] 设置告警规则（P0-P3分级+持续时间阈值）
- [ ] 实现自动降级策略（大模型→小模型→缓存）
- [ ] 建立A/B测试框架（灰度发布+统计检验）
- [ ] 实现OTA更新（增量更新+签名校验+自动回滚）
- [ ] 部署日志分析（结构化日志+性能回归检测）
- [ ] 建立集群监控（在线率/版本分布/全局性能）
- [ ] 构建监控仪表盘（Grafana/Prometheus）
- [ ] 制定运维SOP（故障响应流程+升级机制）